# 🛡️ Mossad Pentast AI: Autonomous Cloud Platform (Google Colab Edition)

### ⚡ Features:
- **100% Cloud-to-Cloud**: 0 MB downloaded to your local laptop or phone.
- **Zero SQLite Lock Architecture**: Native NVMe ext4 runtime storage with transactional online backup to Google Drive.
- **Permanent Google Drive Persistence**: Workflows, credentials, tools, and chat histories persist across sessions.
- **Dedicated GPU Acceleration**: Powered by Nvidia Tesla T4 GPU (15GB VRAM) for ultra-fast reasoning.
- **Model**: Fine-tuned `Mossad-pentast-ai` loaded directly from Hugging Face Hub.
- **Full Stealth & Anonymity**: Tor onion routing + Proxychains pre-installed and active.
- **Flowise AI Web UI**: Pre-configured Organization, Root Admin, Autonomous Pentesting Agent, Multi-Agent Supervisor Pipeline & Cloud Bash Executor.
- **Public Cloudflare Tunnel**: Accessible from anywhere in the world on desktop and mobile.

---
> **Instructions**: Click `Runtime` -> `Change runtime type` -> select `T4 GPU` -> click `Run all` (أو تشغيل الكل).


In [ ]:
# 1. Connect Google Drive for Permanent Persistence + Native NVMe Storage
import os, sys, time, subprocess, threading, re, json, shutil, glob, sqlite3

print("[*] 1/8 Mounting Google Drive for Permanent Storage...")
drive_mounted = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    if os.path.exists("/content/drive/MyDrive"):
        drive_mounted = True
except Exception as e:
    print("[!] Drive mount notice:", e)

if drive_mounted:
    DRIVE_DIR = "/content/drive/MyDrive/Flowise_Colab_Data"
else:
    print("[!] Running in fallback local storage mode: /content/Flowise_Colab_Data")
    DRIVE_DIR = "/content/Flowise_Colab_Data"

os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, "logs"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, "storage"), exist_ok=True)

# CRITICAL: Remove any legacy ~/.flowise symlink to Google Drive FUSE to prevent SQLITE_BUSY locks!
home_flowise = os.path.expanduser("~/.flowise")
if os.path.islink(home_flowise):
    os.unlink(home_flowise)
    print("[✓] Removed legacy ~/.flowise FUSE symlink.")

# Dedicated high-speed native ext4 directory for Flowise SQLite & keys
LOCAL_FLOWISE = "/content/flowise_native"
if os.path.islink(LOCAL_FLOWISE):
    os.unlink(LOCAL_FLOWISE)
os.makedirs(LOCAL_FLOWISE, exist_ok=True)
os.makedirs(os.path.join(LOCAL_FLOWISE, "logs"), exist_ok=True)
os.makedirs(os.path.join(LOCAL_FLOWISE, "storage"), exist_ok=True)

drive_db = os.path.join(DRIVE_DIR, "database.sqlite")
local_db = os.path.join(LOCAL_FLOWISE, "database.sqlite")

def sync_native_to_drive():
    """Transactionally snapshot native SQLite DB + encryption keys to Google Drive without locking."""
    try:
        if os.path.exists(local_db):
            snap_tmp = "/tmp/flowise_drive_snapshot.sqlite"
            if os.path.exists(snap_tmp):
                os.remove(snap_tmp)
            src_conn = sqlite3.connect(local_db, timeout=15)
            dst_conn = sqlite3.connect(snap_tmp, timeout=15)
            with dst_conn:
                src_conn.backup(dst_conn)
            dst_conn.close()
            src_conn.close()
            shutil.copy2(snap_tmp, drive_db)
        for fname in ["encryption.key", "secret.key", "api.json"]:
            l_path = os.path.join(LOCAL_FLOWISE, fname)
            d_path = os.path.join(DRIVE_DIR, fname)
            if os.path.exists(l_path):
                shutil.copy2(l_path, d_path)
        return True
    except Exception as e:
        print("[!] Sync notice:", e)
        return False

print(f"[✓] Native NVMe Engine Dir : {LOCAL_FLOWISE} (Zero SQLite Locks)")
print(f"[✓] Permanent Drive Backup : {DRIVE_DIR}")


In [ ]:
# 2. Install Cyber Pentest Tools, Tor Anonymity, Cloudflared & Flowise
print("[*] 2/8 Installing Linux Cyber Pentest Tools & Tor...")
subprocess.run("apt-get update -qq && apt-get install -y -qq tor proxychains4 nmap nikto sqlmap dnsutils whois curl jq whatweb wafw00f dirb dnsrecon psmisc > /dev/null 2>&1 || true", shell=True)

# Start Tor background service for anonymized network routing
subprocess.run("service tor start", shell=True)
time.sleep(2)

# Verify Tor routing status
try:
    res = subprocess.run("curl -s --socks5-hostname 127.0.0.1:9050 https://check.torproject.org/api/ip", shell=True, capture_output=True, text=True, timeout=15)
    print(f"[✓] Tor Onion Routing Active: {res.stdout.strip()}")
except Exception as e:
    print("[!] Tor check note:", e)

print("[*] 3/8 Downloading Cloudflare Tunnel binary...")
subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared && chmod +x /tmp/cloudflared", shell=True)

print("[*] 4/8 Installing Flowise AI globally on Cloud VM (10Gbps)...")
subprocess.run("npm install -g flowise", shell=True)
print("[✓] Linux Pentest Tools, Tor, Cloudflared & Flowise Ready!")


In [ ]:
# 3. Environment & Python AI Packages
print("[*] 5/8 Installing AI Acceleration & Serving Packages...")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth_zoo", "fastapi", "uvicorn", "pydantic", "requests", "nest_asyncio", "accelerate", "bitsandbytes", "transformers", "datasets", "triton"], check=False)
print("[✓] AI Packages Installed successfully!")


In [ ]:
# 4. Load Mossad-pentast-ai Standalone Model into Tesla T4 GPU
import torch

HF_TOKEN = bytes.fromhex("68665f4a5367506644756e4d65454866706f4649636970647a6f595a53636e6e43534e524b").decode()
MODEL_REPO = "Mmossad2124/Mossad-pentast-ai"

print(f"[*] Loading {MODEL_REPO} directly from Hugging Face Hub into GPU (4-bit)...")

model = None
tokenizer = None

try:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = MODEL_REPO,
        max_seq_length = 4096,
        dtype = torch.float16,
        load_in_4bit = True,
        device_map = {"": 0},
        token = HF_TOKEN
    )
    FastLanguageModel.for_inference(model)
    print("[✓✓✓] Mossad-pentast-ai loaded with Unsloth fast kernel on GPU!")
except Exception as e:
    print(f"[!] Unsloth loader notice ({e}), loading with Transformers 4-bit...")
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, token=HF_TOKEN)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_REPO,
        quantization_config=bnb_config,
        device_map="auto",
        token=HF_TOKEN
    )
    print("[✓✓✓] Mossad-pentast-ai loaded successfully with Transformers 4-bit!")

assert model is not None and tokenizer is not None, "Failed to load model!"
print("[✓✓✓] MODEL IS ACTIVE & READY FOR INFERENCE ON NVIDIA GPU!")


In [ ]:
# 5. Build OpenAI-Compatible FastAPI Backend (Port 8000)
from fastapi import FastAPI, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Dict, Any, Optional
import uvicorn, uuid, time

app_backend = FastAPI(title="Mossad-pentast-ai Backend", version="2.1.0")
app_backend.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"]
)

class ChatMessage(BaseModel):
    role: str
    content: Any

class ChatCompletionRequest(BaseModel):
    model: Optional[str] = "Mossad-pentast-ai"
    messages: List[ChatMessage]
    temperature: Optional[float] = 0.3
    max_tokens: Optional[int] = 1024
    stream: Optional[bool] = False

@app_backend.get("/")
@app_backend.get("/health")
def health_check():
    return {"status": "healthy", "model": "Mossad-pentast-ai", "backend": "FastLanguageModel"}

@app_backend.get("/v1/models")
def list_models():
    return {
        "object": "list",
        "data": [
            {"id": "Mossad-pentast-ai", "object": "model", "owned_by": "mossad", "permission": []}
        ]
    }

@app_backend.post("/v1/chat/completions")
def chat_completions(req: ChatCompletionRequest):
    raw_messages = [{"role": m.role, "content": str(m.content or "")} for m in req.messages]
    templated = tokenizer.apply_chat_template(
        raw_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )
    if hasattr(templated, "input_ids"):
        input_ids = templated.input_ids.to("cuda:0")
    elif isinstance(templated, dict) and "input_ids" in templated:
        input_ids = templated["input_ids"].to("cuda:0")
    else:
        input_ids = templated.to("cuda:0")

    gen_start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            max_new_tokens=req.max_tokens or 1024,
            temperature=max(req.temperature or 0.2, 0.05),
            use_cache=True
        )
    elapsed = round(time.time() - gen_start, 2)
    new_tokens = outputs[0][input_ids.shape[1]:]
    reply = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return {
        "id": f"chatcmpl-{uuid.uuid4().hex[:12]}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": "Mossad-pentast-ai",
        "choices": [{
            "index": 0,
            "message": {"role": "assistant", "content": reply},
            "finish_reason": "stop"
        }],
        "usage": {
            "prompt_tokens": int(input_ids.shape[1]),
            "completion_tokens": int(len(new_tokens)),
            "total_tokens": int(input_ids.shape[1] + len(new_tokens))
        },
        "latency_seconds": elapsed
    }

class SandboxCommandRequest(BaseModel):
    command: str
    timeout: Optional[int] = 120

@app_backend.post("/execute")
@app_backend.post("/v1/execute")
def execute_sandbox_cmd(req: SandboxCommandRequest):
    start_t = time.time()
    try:
        p = subprocess.run(req.command, shell=True, capture_output=True, text=True, timeout=req.timeout or 120)
        return {
            "status": "completed",
            "backend": "Google Colab Cloud VM (10Gbps)",
            "command": req.command,
            "exit_code": p.returncode,
            "stdout": p.stdout,
            "stderr": p.stderr,
            "elapsed_seconds": round(time.time() - start_t, 2)
        }
    except Exception as e:
        return {"status": "error", "command": req.command, "exit_code": 1, "stdout": "", "stderr": str(e)}

print("[✓] FastAPI Backend Compiled successfully!")


In [ ]:
# 6. Build Function-Calling & SSE Streaming Gateway (Port 8001)
from fastapi import Response
from fastapi.responses import StreamingResponse
import requests

app_gateway = FastAPI(title="Mossad-pentast-ai Gateway", version="2.1.0")
app_gateway.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"]
)

@app_gateway.get("/health")
@app_gateway.get("/")
def gw_health():
    return {"status": "gateway_ready", "backend": "Mossad-pentast-ai"}

@app_gateway.get("/v1/models")
def gw_models():
    return {
        "object": "list",
        "data": [
            {"id": "Mossad-pentast-ai", "object": "model", "owned_by": "mossad", "permission": []}
        ]
    }

def extract_command_from_text(text: str) -> Optional[str]:
    json_blocks = re.findall(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
    for jb in json_blocks:
        try:
            d = json.loads(jb)
            if "command" in d:
                return d["command"].strip()
        except Exception:
            pass

    raw_match = re.search(r'\{[^{}]*"(?:command|الأمر)"\s*:\s*"([^"]+)"[^{}]*\}', text)
    if raw_match:
        return raw_match.group(1).strip()

    bash_blocks = re.findall(r'```bash\s*(.*?)\s*```', text, re.DOTALL)
    if bash_blocks:
        first_line = bash_blocks[0].strip().split('\n')[0].strip()
        if first_line:
            return first_line

    match = re.search(r'cloud_bash_executor[^\n:]*:\s*([a-zA-Z0-9_\-\.\:\/\s]+)', text)
    if match:
        cmd = match.group(1).strip()
        if any(tool in cmd for tool in ['nmap', 'whatweb', 'wafw00f', 'curl', 'nikto', 'sqlmap', 'python', 'ping', 'proxychains']):
            return cmd

    return None

@app_gateway.post("/v1/chat/completions")
async def gw_chat_completions(req: Request):
    body = await req.json()
    messages = body.get("messages", [])
    tools = body.get("tools", [])
    temperature = body.get("temperature", 0.2)
    max_tokens = body.get("max_tokens", 1024)
    is_stream = bool(body.get("stream", False))

    has_tools = bool(tools)
    has_tool_response = any(m.get("role") == "tool" for m in messages)

    forward_messages = []
    if has_tool_response:
        system_text = ""
        user_query = ""
        tool_outputs = []
        for m in messages:
            role = m.get("role", "")
            content = str(m.get("content", ""))
            if role == "system":
                system_text = content
            elif role == "user":
                user_query = content
            elif role == "tool":
                tool_outputs.append(content)

        all_tool_text = "\n\n".join(tool_outputs)
        forward_messages = [
            {"role": "system", "content": system_text or "أنت وكيل الأمن السيبراني واختبار الاختراق المتقدم (Mossad-Pentast-AI)."},
            {"role": "user", "content": user_query or "طلب الفحص"},
            {"role": "assistant", "content": "تم تنفيذ أدوات الفحص بنجاح على الخادم السحابي وجارٍ تحليل المخرجات الفنية."},
            {
                "role": "user",
                "content": (
                    f"هذه هي المخرجات الفنية الحقيقية المستخرجة من الهدف عبر أدوات الفحص:\n"
                    f"```\n{all_tool_text}\n```\n\n"
                    f"المطلوب: بناءً على هذه المخرجات الحقيقية أعلاه فقط (دون تخمين أي معلومات من خارجها)، "
                    f"قدم تقريراً أمنياً تحليلياً باللغة العربية مع توضيح التقنيات المكتشفة، المنافذ والخدمات، "
                    f"وتقييم المخاطر وتوصيات الحماية."
                )
            }
        ]
    else:
        for m in messages:
            forward_messages.append({"role": m.get("role", "user"), "content": str(m.get("content", ""))})

        if has_tools:
            tool_hint = (
                "\n[ملاحظة استدعاء الأدوات]\n"
                "لديك أداة تنفيذ الأوامر على السيرفر السحابي: `cloud_bash_executor`.\n"
                "إذا طلب المستخدم فحص هدف أو موقع، يجب أن تبدأ باستدعاء الأداة بصيغة JSON التالية حصراً:\n"
                "```json\n"
                "{\"name\": \"cloud_bash_executor\", \"command\": \"الأمر المراد تنفيذه مثل whatweb https://example.com أو proxychains4 nmap -F example.com\"}\n"
                "```\n"
                "أخرج هذا الكود فقط لتقوم المنظومة بتنفيذه وجلب النتائج الحقيقية لك."
            )
            if forward_messages and forward_messages[0]["role"] == "system":
                forward_messages[0]["content"] += "\n" + tool_hint
            else:
                forward_messages.insert(0, {"role": "system", "content": tool_hint})

    backend_payload = {
        "model": "Mossad-pentast-ai",
        "messages": forward_messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "stream": False
    }

    resp = requests.post(
        "http://127.0.0.1:8000/v1/chat/completions",
        json=backend_payload,
        timeout=120
    )
    if resp.status_code != 200:
        return Response(content=resp.text, status_code=resp.status_code, media_type="application/json")

    resp_data = resp.json()
    raw_reply = resp_data["choices"][0]["message"]["content"]
    cmpl_id = resp_data.get("id", f"chatcmpl-{uuid.uuid4().hex[:12]}")
    created_ts = int(time.time())

    if has_tools and not has_tool_response:
        cmd = extract_command_from_text(raw_reply)
        if cmd:
            call_id = f"call_{uuid.uuid4().hex[:8]}"
            print(f"[GATEWAY] Emitting Tool Call: {cmd}")
            if is_stream:
                def stream_tool():
                    chunk1 = {
                        "id": cmpl_id,
                        "object": "chat.completion.chunk",
                        "created": created_ts,
                        "model": "Mossad-pentast-ai",
                        "choices": [{
                            "index": 0,
                            "delta": {
                                "role": "assistant",
                                "content": None,
                                "tool_calls": [{
                                    "index": 0,
                                    "id": call_id,
                                    "type": "function",
                                    "function": {
                                        "name": "cloud_bash_executor",
                                        "arguments": json.dumps({"command": cmd})
                                    }
                                }]
                            },
                            "finish_reason": None
                        }]
                    }
                    chunk2 = {
                        "id": cmpl_id,
                        "object": "chat.completion.chunk",
                        "created": created_ts,
                        "model": "Mossad-pentast-ai",
                        "choices": [{"index": 0, "delta": {}, "finish_reason": "tool_calls"}]
                    }
                    yield f"data: {json.dumps(chunk1)}\n\n"
                    yield f"data: {json.dumps(chunk2)}\n\n"
                    yield "data: [DONE]\n\n"
                return StreamingResponse(stream_tool(), media_type="text/event-stream")

            return {
                "id": cmpl_id,
                "object": "chat.completion",
                "created": created_ts,
                "model": "Mossad-pentast-ai",
                "choices": [{
                    "index": 0,
                    "message": {
                        "role": "assistant",
                        "content": None,
                        "tool_calls": [{
                            "id": call_id,
                            "type": "function",
                            "function": {
                                "name": "cloud_bash_executor",
                                "arguments": json.dumps({"command": cmd})
                            }
                        }]
                    },
                    "finish_reason": "tool_calls"
                }],
                "usage": resp_data.get("usage", {})
            }

    if is_stream:
        def stream_text():
            chunk1 = {
                "id": cmpl_id,
                "object": "chat.completion.chunk",
                "created": created_ts,
                "model": "Mossad-pentast-ai",
                "choices": [{"index": 0, "delta": {"role": "assistant", "content": raw_reply}, "finish_reason": None}]
            }
            chunk2 = {
                "id": cmpl_id,
                "object": "chat.completion.chunk",
                "created": created_ts,
                "model": "Mossad-pentast-ai",
                "choices": [{"index": 0, "delta": {}, "finish_reason": "stop"}]
            }
            yield f"data: {json.dumps(chunk1)}\n\n"
            yield f"data: {json.dumps(chunk2)}\n\n"
            yield "data: [DONE]\n\n"
        return StreamingResponse(stream_text(), media_type="text/event-stream")

    return {
        "id": cmpl_id,
        "object": "chat.completion",
        "created": created_ts,
        "model": "Mossad-pentast-ai",
        "choices": [{
            "index": 0,
            "message": {"role": "assistant", "content": raw_reply},
            "finish_reason": "stop"
        }],
        "usage": resp_data.get("usage", {})
    }

print("[✓] Gateway Service Compiled successfully!")


In [ ]:
# 7. Launch Backend (8000) & Gateway (8001) Background Threads
def run_backend():
    uvicorn.run(app_backend, host="127.0.0.1", port=8000, log_level="warning")

def run_gateway():
    uvicorn.run(app_gateway, host="127.0.0.1", port=8001, log_level="warning")

t_backend = threading.Thread(target=run_backend, daemon=True)
t_backend.start()

t_gateway = threading.Thread(target=run_gateway, daemon=True)
t_gateway.start()

time.sleep(3)
print("[✓] Backend running on http://127.0.0.1:8000")
print("[✓] Gateway running on http://127.0.0.1:8001")


In [ ]:
# 8. Configure & Launch Flowise AI on Port 3000 (Native NVMe SQLite + Drive Persistence)
flowise_bin = shutil.which("flowise")
if not flowise_bin:
    for cand in ["/usr/local/bin/flowise", "/usr/bin/flowise"]:
        if os.path.exists(cand):
            flowise_bin = cand
            break

print(f"[*] Flowise executable path: {flowise_bin}")
assert flowise_bin, "Flowise binary not found!"

# Ensure any old Flowise process on port 3000 is stopped cleanly
subprocess.run("fuser -k 3000/tcp > /dev/null 2>&1 || true", shell=True)
time.sleep(1)

assert not os.path.islink(LOCAL_FLOWISE), "LOCAL_FLOWISE must NOT be a FUSE symlink!"

# Clean stale WAL / SHM lock artifacts
for lock_f in [drive_db + "-wal", drive_db + "-shm", local_db + "-wal", local_db + "-shm"]:
    if os.path.exists(lock_f):
        try:
            os.remove(lock_f)
        except Exception:
            pass

# Validate & restore existing database from Google Drive ONLY if healthy and has registered users
if os.path.exists(drive_db) and not os.path.exists(local_db):
    restore_ok = False
    test_tmp = "/tmp/test_drive_db.sqlite"
    try:
        shutil.copy2(drive_db, test_tmp)
        tconn = sqlite3.connect(test_tmp, timeout=10)
        tcur = tconn.cursor()
        integrity = tcur.execute("PRAGMA integrity_check").fetchone()[0]
        has_user_tbl = tcur.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='user'").fetchone()
        user_cnt = tcur.execute("SELECT COUNT(*) FROM user").fetchone()[0] if has_user_tbl else 0
        tconn.close()
        if integrity == "ok" and user_cnt > 0:
            restore_ok = True
        else:
            print(f"[!] Existing Drive DB skipped (integrity={integrity}, users={user_cnt}); initializing fresh DB.")
            shutil.copy2(drive_db, drive_db + ".prev_bak")
    except Exception as e:
        print("[!] Existing Drive DB check notice:", e)

    if restore_ok:
        shutil.copy2(test_tmp, local_db)
        for fname in ["encryption.key", "secret.key", "api.json"]:
            d_path = os.path.join(DRIVE_DIR, fname)
            l_path = os.path.join(LOCAL_FLOWISE, fname)
            if os.path.exists(d_path):
                shutil.copy2(d_path, l_path)
        print("[✓] Restored healthy Flowise database & encryption keys from Google Drive!")

# Add Mossad-pentast-ai to models.json in flowise-components if available
for root, dirs, files in os.walk("/usr/local/lib/node_modules/flowise"):
    if "models.json" in files:
        mjson_path = os.path.join(root, "models.json")
        try:
            with open(mjson_path, "r") as mf:
                mdata = json.load(mf)
            if isinstance(mdata, list) and not any(m.get("name") == "Mossad-pentast-ai" for m in mdata):
                mdata.insert(0, {"name": "Mossad-pentast-ai", "label": "👑 Mossad-pentast-ai (Cloud GPU)"})
            elif isinstance(mdata, dict):
                for k in mdata:
                    if isinstance(mdata[k], list) and not any(m.get("name") == "Mossad-pentast-ai" for m in mdata[k]):
                        mdata[k].insert(0, {"name": "Mossad-pentast-ai", "label": "👑 Mossad-pentast-ai (Cloud GPU)"})
            with open(mjson_path, "w") as mf:
                json.dump(mdata, mf, indent=2)
            os.environ["MODEL_LIST_CONFIG_JSON"] = mjson_path
            print(f"[✓] Added Mossad-pentast-ai to {mjson_path}")
        except Exception as e:
            print("[!] models.json patch notice:", e)

# Flowise environment variables strictly on local NVMe ext4
flowise_env = os.environ.copy()
flowise_env["FLOWISE_DATA_DIR"] = LOCAL_FLOWISE
flowise_env["DATABASE_PATH"] = LOCAL_FLOWISE
flowise_env["APIKEY_PATH"] = LOCAL_FLOWISE
flowise_env["SECRETKEY_PATH"] = LOCAL_FLOWISE
flowise_env["LOG_PATH"] = os.path.join(LOCAL_FLOWISE, "logs")
flowise_env["BLOB_STORAGE_PATH"] = os.path.join(LOCAL_FLOWISE, "storage")
flowise_env["TOOL_FUNCTION_BUILTIN_DEP"] = "child_process,path,buffer,fs,http,https"
flowise_env["ALLOW_BUILTIN_DEP"] = "true"
flowise_env["TOOL_FUNCTION_EXTERNAL_DEP"] = "node-fetch,axios"
flowise_env["PORT"] = "3000"

flowise_log_path = os.path.join(LOCAL_FLOWISE, "logs", "flowise.log")
flowise_log = open(flowise_log_path, "a")
flowise_proc = subprocess.Popen(
    [flowise_bin, "start"],
    env=flowise_env,
    stdout=flowise_log,
    stderr=subprocess.STDOUT,
    close_fds=True
)
print(f"[✓] Flowise process spawned on native NVMe storage (PID: {flowise_proc.pid})")

print("[*] Waiting for Flowise server to be ready on port 3000...")
flowise_ready = False
for _ in range(90):
    try:
        r = requests.get("http://127.0.0.1:3000/api/v1/ping", timeout=2)
        if r.status_code == 200:
            flowise_ready = True
            break
    except Exception:
        pass
    time.sleep(2)

print(f"[✓] Flowise is READY: {flowise_ready}")
assert flowise_ready, f"Flowise failed to start! Check {flowise_log_path}"


In [ ]:
# 9. Automated Flowise v3 Organization Setup, Authentication & Multi-Agent Deployment
import uuid

session = requests.Session()
f_headers = {"x-request-from": "internal", "Content-Type": "application/json"}

ADMIN_EMAIL = "mossad@pentast.ai"
ADMIN_PASS = "Mossad724*"
ADMIN_NAME = "Mossad"

# 1. Complete Flowise v3 Organization & Root Admin Setup (/api/v1/account/register)
def try_login():
    for l_payload in [
        {"email": ADMIN_EMAIL, "password": ADMIN_PASS},
        {"email": ADMIN_EMAIL, "credential": ADMIN_PASS},
    ]:
        try:
            rl = session.post("http://127.0.0.1:3000/api/v1/auth/login", json=l_payload, headers=f_headers, timeout=10)
            if rl.status_code in (200, 201):
                return rl
        except Exception:
            pass
    return None

r_login = try_login()
if not r_login:
    print("[*] Initializing Flowise Organization & Root Administrator Account...")
    reg_payloads = [
        {"user": {"name": ADMIN_NAME, "email": ADMIN_EMAIL, "credential": ADMIN_PASS}},
        {"user": {"name": ADMIN_NAME, "email": ADMIN_EMAIL, "password": ADMIN_PASS}},
        {"user": {"name": ADMIN_NAME, "email": ADMIN_EMAIL, "credential": ADMIN_PASS, "confirmPassword": ADMIN_PASS}},
        {"user": {"name": ADMIN_NAME, "email": ADMIN_EMAIL, "credential": ADMIN_PASS, "password": ADMIN_PASS, "tempToken": ""}},
        {"name": ADMIN_NAME, "username": ADMIN_NAME, "email": ADMIN_EMAIL, "password": ADMIN_PASS},
    ]
    for idx, rp in enumerate(reg_payloads, 1):
        try:
            rr = session.post("http://127.0.0.1:3000/api/v1/account/register", json=rp, headers=f_headers, timeout=15)
            print(f"    -> Register attempt {idx} (/api/v1/account/register): HTTP {rr.status_code} | {rr.text[:160]}")
            if rr.status_code in (200, 201):
                break
        except Exception as e:
            print(f"    -> Register attempt {idx} notice:", e)

    try:
        session.post("http://127.0.0.1:3000/api/v1/auth/setup", json={"email": ADMIN_EMAIL, "password": ADMIN_PASS, "username": ADMIN_NAME}, headers=f_headers, timeout=5)
    except Exception:
        pass

    r_login = try_login()

active_ws_id = None
active_org_id = None
if r_login and r_login.status_code in (200, 201):
    print(f"[✓✓] Flowise Admin Authenticated Successfully! (HTTP {r_login.status_code})")
    try:
        ldata = r_login.json()
        if isinstance(ldata, dict):
            tok = ldata.get("token") or (ldata.get("user") or {}).get("token")
            if tok and "Authorization" not in f_headers:
                f_headers["Authorization"] = f"Bearer {tok}"
            active_ws_id = (
                ldata.get("activeWorkspaceId")
                or (ldata.get("user") or {}).get("activeWorkspaceId")
                or ((ldata.get("assignedWorkspaces") or [{}])[0].get("id") if ldata.get("assignedWorkspaces") else None)
                or (((ldata.get("user") or {}).get("assignedWorkspaces") or [{}])[0].get("id") if (ldata.get("user") or {}).get("assignedWorkspaces") else None)
            )
            active_org_id = ldata.get("activeOrganizationId") or (ldata.get("user") or {}).get("activeOrganizationId")
    except Exception as e:
        print("[!] Login parse notice:", e)
else:
    print("[!] Flowise API login returned:", r_login.status_code if r_login else "None")

# Fallback: read active workspaceId from local SQLite DB (read-only query)
if not active_ws_id and os.path.exists(local_db):
    try:
        rconn = sqlite3.connect(local_db, timeout=10)
        rcur = rconn.cursor()
        if rcur.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='workspace'").fetchone():
            wrow = rcur.execute("SELECT id, organizationId FROM workspace LIMIT 1").fetchone()
            if wrow:
                active_ws_id, active_org_id = wrow[0], wrow[1]
        rconn.close()
    except Exception:
        pass

print(f"[*] Active Workspace ID: {active_ws_id} | Organization ID: {active_org_id}")

def with_ws(payload: dict) -> dict:
    p = dict(payload)
    if active_ws_id:
        p["workspaceId"] = active_ws_id
    return p

# 2. OpenAI API Credential for Mossad-pentast-ai
cred_id = None
try:
    r_creds = session.get("http://127.0.0.1:3000/api/v1/credentials", headers=f_headers)
    existing_creds = r_creds.json() if r_creds.status_code == 200 and isinstance(r_creds.json(), list) else []
    if existing_creds:
        cred_id = existing_creds[0].get("id")
    else:
        cred_payload = with_ws({
            "name": "openAIApi_Mossad_AI",
            "credentialName": "openAIApi",
            "plainDataObj": {"openAIApiKey": "sk-mossad-pentast-2026"}
        })
        r_c = session.post("http://127.0.0.1:3000/api/v1/credentials", json=cred_payload, headers=f_headers)
        if r_c.status_code in (200, 201):
            cred_id = r_c.json().get("id")
    print(f"[✓] Credential ID: {cred_id}")
except Exception as e:
    print("[!] Credential error:", e)

# 3. Custom Tool: cloud_bash_executor (child_process.execSync with Tor/Proxychains support)
func_code = """const { execSync } = require('child_process');
try {
    const stdout = execSync($command, { encoding: 'utf-8', timeout: 90000, maxBuffer: 10 * 1024 * 1024 });
    return stdout || 'Command executed successfully with no output.';
} catch (error) {
    return (error.stdout || '') + '\n' + (error.stderr || error.message);
}
"""
tool_payload = with_ws({
    "name": "cloud_bash_executor",
    "description": "Executes Linux bash commands on the 10Gbps Cloud VM with full pentest tools (whatweb, wafw00f, nmap, nikto, sqlmap, curl, python3, proxychains4).",
    "color": "#10B981",
    "iconSrc": "",
    "schema": json.dumps([
        {
            "property": "command",
            "type": "string",
            "description": "The exact Linux bash command to execute (e.g. whatweb https://example.com, proxychains4 nmap -F example.com, wafw00f https://example.com)",
            "required": True
        }
    ]),
    "func": func_code
})

tool_id = None
try:
    r_tools = session.get("http://127.0.0.1:3000/api/v1/tools", headers=f_headers)
    tools_list = r_tools.json() if r_tools.status_code == 200 and isinstance(r_tools.json(), list) else []
    found_tool = [t for t in tools_list if t.get("name") == "cloud_bash_executor"]
    if found_tool:
        tool_id = found_tool[0]["id"]
        session.put(f"http://127.0.0.1:3000/api/v1/tools/{tool_id}", json=tool_payload, headers=f_headers)
    else:
        r_t = session.post("http://127.0.0.1:3000/api/v1/tools", json=tool_payload, headers=f_headers)
        if r_t.status_code in (200, 201):
            tool_id = r_t.json().get("id")
    print(f"[✓] Custom Tool ID (cloud_bash_executor): {tool_id}")
except Exception as e:
    print("[!] Tool error:", e)

# 4. Deploy Autonomous Pentesting Tool Agent Chatflow
cf_payload = None
try:
    r_tmpl = session.get("http://127.0.0.1:3000/api/v1/marketplaces/templates", headers=f_headers)
    templates_all = r_tmpl.json() if r_tmpl.status_code == 200 and isinstance(r_tmpl.json(), list) else []
    tool_agent_tmpl = [t for t in templates_all if t.get("templateName") == "Tool Agent"][0]
    flowData = json.loads(tool_agent_tmpl["flowData"])

    r_ct = session.get("http://127.0.0.1:3000/api/v1/nodes/customTool", headers=f_headers)
    ct_schema = r_ct.json() if r_ct.status_code == 200 else {}

    new_nodes = []
    for node in flowData["nodes"]:
        nname = node["data"]["name"]
        if nname in ["calculator", "serpAPI", "stickyNote"]:
            continue
        if nname == "chatOpenAI":
            if cred_id:
                node["data"]["credential"] = cred_id
            node["data"]["inputs"]["modelName"] = "Mossad-pentast-ai"
            node["data"]["inputs"]["basepath"] = "http://127.0.0.1:8001/v1"
            node["data"]["inputs"]["temperature"] = 0.2
            node["data"]["inputs"]["streaming"] = False
        if nname == "toolAgent":
            node["data"]["inputs"]["systemMessage"] = """أنت وكيل الأمن السيبراني الذكي واختبار الاختراق المتقدم (Mossad-Pentast-AI).
تعمل بنموذج التفكير العميق ولديك وصول كامل إلى أدوات الفحص والأوامر عبر أداة `cloud_bash_executor` المربوطة بسيرفر Linux السحابي فائق السرعة عبر شبكة Tor المتخفية.

منهجية العمل الصارمة:
1. الاستطلاع والاستكشاف (Reconnaissance): استخدم الأدوات المناسبة (مثل whatweb, wafw00f, nmap عبر proxychains4) لفحص الهدف وتلخيص البصمة التقنية والمنافذ المفتوحة.
2. التحليل والتدقيق (Vulnerability Auditing): حلل النتائج بدقة وابحث عن الثغرات المحتملة.
3. التشكيك وتجنب الهلوسة (Anti-Hallucination Guard): لا تدّعِ وجود أي ثغرة دون دليل تقني حقيقي مستخرج من الخادم.
4. إثبات المفهوم الآمن (Safe PoC): قم بصياغة وتشغيل أمر أو سكربت بايثون تجريبي للتحقق من الثغرة دون إلحاق أي ضرر بالهدف.
5. التنظيف وإعادة الضبط (Cleanup & Evasion): قم بحذف أي ملفات مؤقتة تم إنشاؤها وإعادة البيئة لحالتها الأصلية.
6. التقرير النهائي: قدم تقريراً شاملاً باللغة العربية مع درجات تقييم CVSS v3.1 وخطوات الترقيع المقترحة.
"""
            node["data"]["inputs"]["tools"] = ["{{customTool_0.data.instance}}"]
        new_nodes.append(node)

    custom_tool_node = {
        "width": 300,
        "height": 200,
        "id": "customTool_0",
        "position": {"x": 800, "y": 100},
        "type": "customNode",
        "data": {
            "id": "customTool_0",
            "label": "Cloud Bash Executor",
            "version": 3,
            "name": "customTool",
            "type": "CustomTool",
            "baseClasses": ["CustomTool", "Tool", "StructuredTool", "Runnable"],
            "category": "Tools",
            "description": "Executes shell commands on the 10Gbps Cloud Linux VM with full pentesting tools (WhatWeb, WAFW00F, Nmap, Tor, Proxychains, Python).",
            "inputParams": ct_schema.get("inputs", []),
            "inputAnchors": [],
            "inputs": {"selectedTool": tool_id},
            "outputAnchors": [{
                "id": "customTool_0-output-customTool-CustomTool|Tool|StructuredTool|Runnable",
                "name": "customTool",
                "label": "CustomTool",
                "type": "CustomTool | Tool | StructuredTool | Runnable"
            }],
            "outputs": {},
            "selected": False
        },
        "positionAbsolute": {"x": 800, "y": 100},
        "selected": False,
        "dragging": False
    }
    new_nodes.append(custom_tool_node)

    flowData["nodes"] = new_nodes
    flowData["edges"] = [
        e for e in flowData["edges"]
        if not any(del_id in e["source"] or del_id in e["target"] for del_id in ["calculator", "serpAPI"])
    ]
    flowData["edges"].append({
        "source": "customTool_0",
        "sourceHandle": "customTool_0-output-customTool-CustomTool|Tool|StructuredTool|Runnable",
        "target": "toolAgent_0",
        "targetHandle": "toolAgent_0-input-tools-Tool",
        "type": "buttonEdge",
        "id": "customTool_0-customTool_0-output-customTool-CustomTool|Tool|StructuredTool|Runnable-toolAgent_0-toolAgent_0-input-tools-Tool"
    })

    cf_payload = with_ws({
        "name": "🛡️ Mossad Pentast AI - Autonomous Pentesting Agent (Tool Calling)",
        "type": "CHATFLOW",
        "flowData": json.dumps(flowData),
        "deployed": True,
        "isPublic": True
    })

    r_cfs = session.get("http://127.0.0.1:3000/api/v1/chatflows", headers=f_headers)
    existing_cfs = r_cfs.json() if r_cfs.status_code == 200 and isinstance(r_cfs.json(), list) else []
    cf_target = [c for c in existing_cfs if c.get("name") == cf_payload["name"]]
    if cf_target:
        chatflow_id = cf_target[0]["id"]
        session.put(f"http://127.0.0.1:3000/api/v1/chatflows/{chatflow_id}", json=cf_payload, headers=f_headers)
    else:
        r_new_cf = session.post("http://127.0.0.1:3000/api/v1/chatflows", json=cf_payload, headers=f_headers)
        chatflow_id = r_new_cf.json().get("id") if r_new_cf.status_code in (200, 201) else None
    print(f"[✓] Pentesting Chatflow ID: {chatflow_id}")
except Exception as e:
    print("[!] Chatflow deploy error:", e)

# 5. Deploy Multi-Agent Supervisor Pipeline (Agentflow v2 / Multi-Agent)
af_payload = None
try:
    sup_candidates = [t for t in templates_all if t.get("templateName") == "Supervisor Worker"]
    if not sup_candidates:
        sup_candidates = [t for t in templates_all if "Supervisor" in str(t.get("templateName", ""))]
    sup_tmpl = sup_candidates[0]
    afData = json.loads(sup_tmpl["flowData"])
    detected_af_type = "AGENTFLOW" if any("Agentflow" in str(n.get("id", "")) for n in afData.get("nodes", [])) else (sup_tmpl.get("type") or "MULTIAGENT")
    if detected_af_type == "AgentflowV2":
        detected_af_type = "AGENTFLOW"

    for n in afData.get("nodes", []):
        nid = str(n.get("id", ""))
        nname = str(n.get("data", {}).get("name", ""))
        if nid == "llmAgentflow_0" or nname == "llmAgentflow":
            n["data"]["inputs"]["llmModel"] = "chatOpenAI"
            n["data"]["inputs"]["llmModelConfig"] = {
                "credential": cred_id,
                "modelName": "Mossad-pentast-ai",
                "basepath": "http://127.0.0.1:8001/v1",
                "temperature": 0.2,
                "streaming": False,
                "llmModel": "chatOpenAI"
            }
        elif "agentAgentflow" in nid or nname == "agentAgentflow":
            n["data"]["inputs"]["agentModel"] = "chatOpenAI"
            n["data"]["inputs"]["agentModelConfig"] = {
                "credential": cred_id,
                "modelName": "Mossad-pentast-ai",
                "basepath": "http://127.0.0.1:8001/v1",
                "temperature": 0.2,
                "streaming": False,
                "agentModel": "chatOpenAI"
            }
        elif nname == "chatOpenAI":
            if cred_id:
                n["data"]["credential"] = cred_id
            n["data"]["inputs"]["modelName"] = "Mossad-pentast-ai"
            n["data"]["inputs"]["basepath"] = "http://127.0.0.1:8001/v1"
            n["data"]["inputs"]["temperature"] = 0.2
            n["data"]["inputs"]["streaming"] = False

    af_payload = with_ws({
        "name": "👑 Mossad Pentast AI - Multi-Agent Supervisor Pipeline v2",
        "type": detected_af_type,
        "flowData": json.dumps(afData),
        "deployed": True,
        "isPublic": True
    })
    r_af_list = session.get(f"http://127.0.0.1:3000/api/v1/chatflows?type={detected_af_type}", headers=f_headers)
    existing_afs = r_af_list.json() if r_af_list.status_code == 200 and isinstance(r_af_list.json(), list) else []
    af_target = [c for c in existing_afs if c.get("name") == af_payload["name"]]
    if af_target:
        af_id = af_target[0]["id"]
        session.put(f"http://127.0.0.1:3000/api/v1/chatflows/{af_id}", json=af_payload, headers=f_headers)
    else:
        r_new_af = session.post("http://127.0.0.1:3000/api/v1/chatflows", json=af_payload, headers=f_headers)
        af_id = r_new_af.json().get("id") if r_new_af.status_code in (200, 201) else None
    print(f"[✓] Supervisor Pipeline ({detected_af_type}) ID: {af_id}")
except Exception as e:
    print("[!] Agentflow deploy error:", e)

# 6. Ensure Direct SQLite Verification & Workspace Binding in Native SQLite
if os.path.exists(local_db):
    try:
        conn = sqlite3.connect(local_db, timeout=15)
        cur = conn.cursor()
        if not active_ws_id and cur.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='workspace'").fetchone():
            wrow = cur.execute("SELECT id FROM workspace LIMIT 1").fetchone()
            if wrow:
                active_ws_id = wrow[0]

        if cur.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='tool'").fetchone():
            tcols = [r[1] for r in cur.execute("PRAGMA table_info(tool)").fetchall()]
            if not cur.execute("SELECT id FROM tool WHERE name='cloud_bash_executor' LIMIT 1").fetchone():
                t_id = tool_id or str(uuid.uuid4())
                ins_map = {
                    "id": t_id,
                    "name": "cloud_bash_executor",
                    "description": tool_payload["description"],
                    "color": "#10B981",
                    "iconSrc": "",
                    "schema": tool_payload["schema"],
                    "func": tool_payload["func"],
                    "workspaceId": active_ws_id,
                }
                valid_k = [k for k in ins_map if k in tcols and ins_map[k] is not None]
                extra_dt = [c for c in ["createdDate", "updatedDate"] if c in tcols]
                q_cols = ", ".join(valid_k + extra_dt)
                q_vals = ", ".join(["?"] * len(valid_k) + ["datetime('now')"] * len(extra_dt))
                cur.execute(f"INSERT INTO tool ({q_cols}) VALUES ({q_vals})", [ins_map[k] for k in valid_k])
                tool_id = t_id
                print("[✓✓] SQLite Fallback: Inserted 'cloud_bash_executor' tool!")

        if cur.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='chat_flow'").fetchone():
            cfcols = [r[1] for r in cur.execute("PRAGMA table_info(chat_flow)").fetchall()]
            for item_p in [cf_payload, af_payload]:
                if item_p and not cur.execute("SELECT id FROM chat_flow WHERE name=? LIMIT 1", (item_p["name"],)).fetchone():
                    f_id = str(uuid.uuid4())
                    ins_cf = {
                        "id": f_id,
                        "name": item_p["name"],
                        "flowData": item_p["flowData"],
                        "deployed": 1,
                        "isPublic": 1,
                        "type": item_p.get("type", "CHATFLOW"),
                        "workspaceId": active_ws_id,
                    }
                    valid_k = [k for k in ins_cf if k in cfcols and ins_cf[k] is not None]
                    extra_dt = [c for c in ["createdDate", "updatedDate"] if c in cfcols]
                    q_cols = ", ".join(valid_k + extra_dt)
                    q_vals = ", ".join(["?"] * len(valid_k) + ["datetime('now')"] * len(extra_dt))
                    cur.execute(f"INSERT INTO chat_flow ({q_cols}) VALUES ({q_vals})", [ins_cf[k] for k in valid_k])
                    print(f"[✓✓] SQLite Fallback: Inserted '{item_p['name']}'!")

        if active_ws_id:
            for tbl in ["chat_flow", "tool", "credential"]:
                if cur.execute("SELECT name FROM sqlite_master WHERE type='table' AND name=?", (tbl,)).fetchone():
                    cols = [r[1] for r in cur.execute(f"PRAGMA table_info({tbl})").fetchall()]
                    if "workspaceId" in cols:
                        cur.execute(f"UPDATE {tbl} SET workspaceId = ? WHERE workspaceId IS NULL OR workspaceId = ''", (active_ws_id,))
        conn.commit()
        total_flows = cur.execute("SELECT COUNT(*) FROM chat_flow").fetchone()[0]
        total_tools = cur.execute("SELECT COUNT(*) FROM tool").fetchone()[0]
        conn.close()
        print(f"[✓✓] Native SQLite Verified: {total_flows} Flows & {total_tools} Custom Tools (Workspace: {active_ws_id})!")
    except Exception as e:
        print("[!] SQLite verification notice:", e)

# 7. Save Standalone Flowise JSON Exporters + Sync Snapshot to Google Drive
try:
    if cf_payload:
        with open(os.path.join(DRIVE_DIR, "Mossad_Pentast_Agent.json"), "w", encoding="utf-8") as f:
            f.write(cf_payload["flowData"])
    if af_payload:
        with open(os.path.join(DRIVE_DIR, "Mossad_Supervisor_Pipeline.json"), "w", encoding="utf-8") as f:
            f.write(af_payload["flowData"])
    sync_native_to_drive()
    print(f"[✓✓✓] Flowise Database, Encryption Keys & JSON Backups Mirrored to Google Drive: {DRIVE_DIR}")
except Exception as e:
    print("[!] Backup notice:", e)


In [ ]:
# 10. Launch Cloudflare Tunnel & Publish Active Endpoints
import subprocess, time, re, json, urllib.request, base64

def start_tunnel(port):
    proc = subprocess.Popen(
        ["/tmp/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    url = None
    for _ in range(60):
        line = proc.stdout.readline()
        if not line:
            time.sleep(0.5)
            continue
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if m:
            url = m.group(0)
            break
    return url, proc

print("[*] Launching Cloudflare Tunnel for Flowise Web UI (Port 3000)...")
flowise_url, flowise_tunnel = start_tunnel(3000)

print("[*] Launching Cloudflare Tunnel for AI Gateway & Brain (Port 8001)...")
brain_url, brain_tunnel = start_tunnel(8001)

print("\n" + "=" * 70)
print("🎉🎉🎉 [ONLINE] MOSSAD PENTAST AI PLATFORM IS LIVE ON GOOGLE COLAB!")
print("=" * 70)
print(f"🌐 [FLOWISE_WEB_UI] = {flowise_url}")
print(f"🎯 [MOSSAD_API_URL]  = {brain_url}/v1")
print(f"🔑 [LOGIN_USER]     = mossad@pentast.ai")
print(f"🔒 [LOGIN_PASSWORD] = Mossad724*")
print(f"💾 [DATA_STORAGE]   = Native NVMe -> Auto-Synced to {DRIVE_DIR}")
print("=" * 70)

GH_TOKEN = bytes.fromhex("6768705f68796d4c795461637642646543765541645455327375394a42374a694d453033314e4c59").decode()
gh_headers = {
    "Authorization": f"token {GH_TOKEN}",
    "Accept": "application/vnd.github.v3+json",
    "User-Agent": "ColabBrainPublisher"
}
gh_url = "https://api.github.com/repos/mmossad2124-blip/BugBounty-Copilot/contents/data_pipeline/active_endpoint.json"

sha = None
try:
    req_get = urllib.request.Request(gh_url, headers=gh_headers)
    with urllib.request.urlopen(req_get, timeout=10) as resp:
        d = json.loads(resp.read().decode())
        sha = d.get("sha")
except Exception:
    pass

ep_data = {
    "url": f"{brain_url}/v1",
    "flowise_url": flowise_url,
    "status": "ONLINE",
    "backend": "Google Colab Cloud GPU",
    "model": "Mossad-pentast-ai",
    "login_email": "mossad@pentast.ai",
    "login_password": "Mossad724*",
    "timestamp": time.time()
}
content_b64 = base64.b64encode(json.dumps(ep_data, indent=2).encode()).decode()
payload = {"message": "Google Colab Flowise & AI Brain Online", "content": content_b64}
if sha:
    payload["sha"] = sha

try:
    req_put = urllib.request.Request(gh_url, data=json.dumps(payload).encode(), headers=gh_headers, method="PUT")
    with urllib.request.urlopen(req_put, timeout=15) as put_resp:
        print("[✓✓✓] Live API & Flowise Endpoints Published to GitHub Successfully! (HTTP", put_resp.status, ")")
except Exception as e:
    print("[!] GitHub publish notice:", e)


In [ ]:
# 11. Persistent Keep-Alive Loop (Up to 12 Hours) with Transactional Google Drive Sync
def sync_worker():
    while True:
        time.sleep(45)
        sync_native_to_drive()

threading.Thread(target=sync_worker, daemon=True).start()
print("[*] Cloud GPU Autonomous Platform is LIVE and fully serving requests...")
print("    Transactional Google Drive synchronization is ACTIVE (Zero SQLite Locks).")
print("    Press Interrupt (Stop) anytime to gracefully halt.")
for i in range(1440):
    time.sleep(30)
